In [2]:
# -------------------------------------------------------------------------------------
# Dashboard de Análisis de Estrategias v2.0 - Versión Corregida y Optimizada
# -------------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# ======================================================================================
# CONFIGURACIÓN
# ======================================================================================

class Config:
    """Configuración centralizada del dashboard"""
    # Dimensiones de gráficos
    SCATTER_HEIGHT = 600
    SCATTER_WIDTH = 550
    HEATMAP_HEIGHT = 1200
    HEATMAP_WIDTH = 1700
    PNL_HEIGHT = 1200
    PNL_WIDTH = 1700
    
    # Archivos de datos
    ANALYSIS_FILE = 'analisis_final_automatizado.xlsx'
    TRADES_FILE = 'operaciones_maestro_combinado.csv'
    
    # Estilos de tabla
    TABLE_STYLES = [
        {'selector': 'th', 'props': [('background-color', '#333'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('color', 'white'), ('border-color', '#555')]},
        {'selector': '', 'props': [('background-color', '#222')]}
    ]

# ======================================================================================
# CARGA DE DATOS
# ======================================================================================

def cargar_datos():
    """Carga y valida los archivos de datos necesarios"""
    try:
        df_results = pd.read_excel(Config.ANALYSIS_FILE)
        df_trades = pd.read_csv(Config.TRADES_FILE)
        df_trades['timestamp'] = pd.to_datetime(df_trades['timestamp'])
        print(f"✓ Datos cargados: {len(df_results)} estrategias, {len(df_trades)} operaciones")
        return df_results, df_trades
    except FileNotFoundError as e:
        print(f"✗ Error: No se encontró el archivo {e.filename}")
        return None, None
    except Exception as e:
        print(f"✗ Error al cargar datos: {str(e)}")
        return None, None

# ======================================================================================
# MOTOR DE SIMULACIÓN
# ======================================================================================

def simular_curva_capital(stop_loss, take_profit, df_trades):
    """
    Simula la curva de capital basada en los recorridos de precio
    
    Args:
        stop_loss: Stop Loss en pips
        take_profit: Take Profit en pips
        df_trades: DataFrame con las operaciones y sus recorridos
    
    Returns:
        dict: {'x': array de índices, 'y': array de capital acumulado}
    """
    pnl_history = []
    
    for recorrido_str in df_trades['Recorrido']:
        # Validar datos
        if not isinstance(recorrido_str, str) or pd.isna(recorrido_str) or recorrido_str == '':
            pnl_history.append(-stop_loss)
            continue
        
        # Inicializar resultado como pérdida
        resultado = -stop_loss
        eventos = recorrido_str.split(',')
        
        # Procesar cada evento del recorrido
        for evento in eventos:
            if not evento:
                continue
                
            tipo = evento[0].upper()
            try:
                valor = int(evento[1:])
            except (ValueError, IndexError):
                continue
            
            # Verificar condiciones de salida
            if tipo == 'M' and valor >= take_profit:
                resultado = take_profit
                break
            elif tipo == 'D' and valor >= stop_loss:
                resultado = -stop_loss
                break
        
        pnl_history.append(resultado)
    
    # Calcular capital acumulado
    capital_acumulado = np.cumsum([0] + pnl_history)
    
    return {
        'x': np.arange(len(capital_acumulado)),
        'y': capital_acumulado,
        'num_trades': len(pnl_history)
    }

# ======================================================================================
# CLASE PRINCIPAL DEL DASHBOARD
# ======================================================================================

class TradingDashboard:
    """Dashboard interactivo para análisis de estrategias de trading"""
    
    def __init__(self, df_results, df_trades):
        self.df_original = df_results
        self.df_trades = df_trades
        self.df_display = df_results.copy()
        
        # Crear componentes de UI
        self._crear_widgets()
        self._crear_graficos()
        self._configurar_callbacks()
    
    def _crear_widgets(self):
        """Crea todos los widgets de control"""
        # Menús de ordenamiento
        self.sort_column = widgets.Dropdown(
            options=list(self.df_original.columns),
            value='Composite_Score',
            description='Ordenar por:',
            style={'description_width': 'initial'}
        )
        
        self.sort_order = widgets.RadioButtons(
            options=['Descendente', 'Ascendente'],
            value='Descendente',
            description='Orden:'
        )
        
        # Sliders de filtrado
        self.filter_widgets = {}
        numeric_cols = self.df_original.select_dtypes(include=['number']).columns
        
        for col in numeric_cols:
            min_val = float(self.df_original[col].min())
            max_val = float(self.df_original[col].max())
            
            if min_val == max_val:
                continue
            
            # Determinar tipo de slider
            is_integer = pd.api.types.is_integer_dtype(self.df_original[col].dtype)
            slider_class = widgets.IntRangeSlider if is_integer else widgets.FloatRangeSlider
            step = 1 if is_integer else (max_val - min_val) / 100
            
            self.filter_widgets[col] = slider_class(
                value=[min_val, max_val],
                min=min_val,
                max=max_val,
                step=step,
                description=f'{col}:',
                layout={'width': '800px'},
                readout_format='.2f' if not is_integer else '.0f',
                continuous_update=False,
                style={'description_width': '200px'}
            )
        
        # Output para tabla
        self.table_output = widgets.Output()
    
    def _crear_graficos(self):
        """Crea todos los gráficos interactivos"""
        # Scatter Plot 1: Color = Profit Factor
        self.g_scatter_pf = go.FigureWidget(
            layout={
                'template': 'plotly_dark',
                'height': Config.SCATTER_HEIGHT,
                'width': Config.SCATTER_WIDTH,
                'title': 'Análisis R/R (Color = Profit Factor)',
                'xaxis_title': 'Max Drawdown',
                'yaxis_title': 'Net P/L'
            }
        )
        
        # Scatter Plot 2: Color = Composite Score
        self.g_scatter_cs = go.FigureWidget(
            layout={
                'template': 'plotly_dark',
                'height': Config.SCATTER_HEIGHT,
                'width': Config.SCATTER_WIDTH,
                'title': 'Análisis R/R (Color = Composite Score)',
                'xaxis_title': 'Max Drawdown',
                'yaxis_title': 'Net P/L'
            }
        )
        
        # Heatmap
        self.g_heatmap = go.FigureWidget(
            layout={
                'template': 'plotly_dark',
                'height': Config.HEATMAP_HEIGHT,
                'width': Config.HEATMAP_WIDTH,
                'title': 'Mapa de Calor (Color = Recovery Factor)',
                'xaxis_title': 'Take Profit',
                'yaxis_title': 'Stop Loss'
            }
        )
        
        # Curva de Capital
        self.g_pnl_curve = go.FigureWidget(
            layout={
                'template': 'plotly_dark',
                'height': Config.PNL_HEIGHT,
                'width': Config.PNL_WIDTH,
                'title': 'Curva de Capital - Selecciona una estrategia',
                'xaxis_title': 'Número de Operación',
                'yaxis_title': 'Capital Acumulado'
            }
        )
    
    def _configurar_callbacks(self):
        """Configura todos los callbacks de interactividad"""
        # Callbacks para filtros y ordenamiento
        for widget in [self.sort_column, self.sort_order, *self.filter_widgets.values()]:
            widget.observe(self._on_filter_change, names='value')
        
        # CORRECCIÓN CRÍTICA: Usar batch_update para configurar callbacks después de añadir datos
        # Los callbacks se configurarán en _actualizar_visualizaciones()
    
    def _on_filter_change(self, change):
        """Callback cuando cambia algún filtro o criterio de ordenamiento"""
        # Aplicar filtros
        df_filtered = self.df_original.copy()
        
        for col, slider in self.filter_widgets.items():
            min_val, max_val = slider.value
            df_filtered = df_filtered[
                (df_filtered[col] >= min_val) & 
                (df_filtered[col] <= max_val)
            ]
        
        # Ordenar
        ascending = (self.sort_order.value == 'Ascendente')
        self.df_display = df_filtered.sort_values(
            by=self.sort_column.value,
            ascending=ascending
        ).reset_index(drop=True)
        
        # Actualizar visualizaciones
        self._actualizar_tabla()
        self._actualizar_visualizaciones()
    
    def _actualizar_tabla(self):
        """Actualiza la tabla de resultados"""
        with self.table_output:
            clear_output(wait=True)
            if len(self.df_display) == 0:
                print("No hay estrategias que cumplan los criterios actuales")
            else:
                display(
                    self.df_display.head(10).style
                    .set_table_styles(Config.TABLE_STYLES)
                    .format(precision=2)
                )
    
    def _actualizar_visualizaciones(self):
        """Actualiza todos los gráficos con los datos filtrados"""
        if len(self.df_display) == 0:
            # Limpiar gráficos si no hay datos
            self.g_scatter_pf.data = []
            self.g_scatter_cs.data = []
            self.g_heatmap.data = []
            return
        
        # Preparar datos personalizados para hover
        hovertemplate = (
            "<b>Stop: %{customdata[0]}, Profit: %{customdata[1]}</b><br>"
            "Net P/L: %{y:.0f}<br>"
            "Max DD: %{x:.0f}<br>"
            "Win Rate: %{customdata[2]:.2f}%<br>"
            "PF: %{customdata[3]:.2f}<br>"
            "Score: %{customdata[4]:.2f}"
            "<extra></extra>"
        )
        
        custom_data = self.df_display[[
            'Stop', 'Profit', 'Win Rate (%)', 
            'Profit Factor', 'Composite_Score'
        ]].values
        
        # Actualizar Scatter Plot 1 (Profit Factor)
        with self.g_scatter_pf.batch_update():
            self.g_scatter_pf.data = []
            scatter_pf = go.Scatter(
                x=self.df_display['Max Drawdown'],
                y=self.df_display['Net P/L'],
                mode='markers',
                marker=dict(
                    color=self.df_display['Profit Factor'],
                    colorscale='Plasma',
                    showscale=True,
                    size=8,
                    colorbar=dict(title='Profit Factor')
                ),
                customdata=custom_data,
                hovertemplate=hovertemplate
            )
            self.g_scatter_pf.add_trace(scatter_pf)
            
            # CORRECCIÓN: Configurar callback después de añadir el trace
            self.g_scatter_pf.data[0].on_click(self._on_scatter_click)
        
        # Actualizar Scatter Plot 2 (Composite Score)
        with self.g_scatter_cs.batch_update():
            self.g_scatter_cs.data = []
            scatter_cs = go.Scatter(
                x=self.df_display['Max Drawdown'],
                y=self.df_display['Net P/L'],
                mode='markers',
                marker=dict(
                    color=self.df_display['Composite_Score'],
                    colorscale='Viridis',
                    showscale=True,
                    size=8,
                    colorbar=dict(title='Composite Score')
                ),
                customdata=custom_data,
                hovertemplate=hovertemplate
            )
            self.g_scatter_cs.add_trace(scatter_cs)
            
            # CORRECCIÓN: Configurar callback después de añadir el trace
            self.g_scatter_cs.data[0].on_click(self._on_scatter_click)
        
        # Actualizar Heatmap
        with self.g_heatmap.batch_update():
            self.g_heatmap.data = []
            
            # Crear pivot table
            pivot_data = self.df_display.pivot_table(
                index='Stop',
                columns='Profit',
                values='Recovery Factor'
            ).fillna(0)
            
            heatmap = go.Heatmap(
                z=pivot_data.values,
                x=pivot_data.columns,
                y=pivot_data.index,
                colorscale='Viridis',
                colorbar=dict(title='Recovery Factor'),
                hovertemplate=(
                    "TP: %{x}<br>"
                    "SL: %{y}<br>"
                    "RF: %{z:.2f}"
                    "<extra></extra>"
                )
            )
            self.g_heatmap.add_trace(heatmap)
            
            # CORRECCIÓN: Configurar callback después de añadir el trace
            self.g_heatmap.data[0].on_click(self._on_heatmap_click)
    
    def _on_scatter_click(self, trace, points, state):
        """Callback cuando se hace clic en un punto del scatter plot"""
        if not points.point_inds:
            return
        
        # Obtener índice del punto clickeado
        idx = points.point_inds[0]
        
        # Obtener estrategia seleccionada
        selected = self.df_display.iloc[idx]
        stop = int(selected['Stop'])
        profit = int(selected['Profit'])
        
        # Actualizar curva de capital
        self._actualizar_curva_capital(stop, profit)
    
    def _on_heatmap_click(self, trace, points, state):
        """Callback cuando se hace clic en una celda del heatmap"""
        if not (hasattr(points, 'xs') and hasattr(points, 'ys')):
            return
        
        # Obtener coordenadas de la celda clickeada
        profit = int(points.xs[0])
        stop = int(points.ys[0])
        
        # Actualizar curva de capital
        self._actualizar_curva_capital(stop, profit)
    
    def _actualizar_curva_capital(self, stop, profit):
        """Actualiza el gráfico de curva de capital"""
        try:
            # Simular curva
            curve_data = simular_curva_capital(stop, profit, self.df_trades)
            
            # Actualizar gráfico
            with self.g_pnl_curve.batch_update():
                self.g_pnl_curve.data = []
                self.g_pnl_curve.add_scatter(
                    x=curve_data['x'],
                    y=curve_data['y'],
                    mode='lines',
                    line=dict(color="#00CF9B", width=2),
                    name=f'SL={stop}, TP={profit}'
                )
                
                # Actualizar título con información adicional
                final_pnl = curve_data['y'][-1]
                num_trades = curve_data['num_trades']
                self.g_pnl_curve.layout.title = (
                    f'Curva de Capital: Stop={stop}, Profit={profit} | '
                    f'P/L Final: {final_pnl:.0f} | '
                    f'Operaciones: {num_trades}'
                )
        except Exception as e:
            print(f"Error al generar curva de capital: {str(e)}")
    
    def construir_ui(self):
        """Construye y retorna la interfaz de usuario completa"""
        # Panel de ordenamiento
        sorting_box = widgets.HBox([self.sort_column, self.sort_order])
        
        # Separar filtros en categorías
        main_filters = [
            w for k, w in self.filter_widgets.items() 
            if '_score' not in k.lower()
        ]
        score_filters = [
            w for k, w in self.filter_widgets.items() 
            if '_score' in k.lower()
        ]
        
        # Crear pestañas de filtros
        filter_tabs = widgets.Tab(children=[
            widgets.VBox(main_filters),
            widgets.VBox(score_filters)
        ])
        filter_tabs.set_title(0, 'Métricas Principales')
        filter_tabs.set_title(1, 'Scores Ponderados')
        
        # Pestañas de visualizaciones
        scatter_box = widgets.HBox([self.g_scatter_pf, self.g_scatter_cs])
        graph_tabs = widgets.Tab(children=[scatter_box, self.g_heatmap])
        graph_tabs.set_title(0, 'Análisis de Dispersión')
        graph_tabs.set_title(1, 'Mapa de Calor')
        
        # Layout principal
        ui = widgets.VBox([
            widgets.HTML("<h1>📊 Dashboard de Análisis de Estrategias</h1>"),
            widgets.HTML("<hr>"),
            widgets.HTML("<h3>🎛️ Panel de Control</h3>"),
            sorting_box,
            filter_tabs,
            widgets.HTML("<hr>"),
            widgets.HTML("<h3>📋 Top 10 Estrategias Filtradas</h3>"),
            self.table_output,
            widgets.HTML("<hr>"),
            widgets.HTML("<h3>📈 Visualizaciones Interactivas</h3>"),
            graph_tabs,
            widgets.HTML("<hr>"),
            widgets.HTML("<h3>💹 Curva de Capital (Haz clic en un gráfico)</h3>"),
            self.g_pnl_curve
        ])
        
        return ui

# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

def main():
    """Función principal para inicializar el dashboard"""
    print("🚀 Iniciando Dashboard de Análisis de Estrategias v2.0...")
    print("-" * 60)
    
    # Cargar datos
    df_results, df_trades = cargar_datos()
    
    if df_results is None or df_trades is None:
        print("\n❌ No se pudo inicializar el dashboard debido a errores en la carga de datos")
        return
    
    # Crear dashboard
    print("\n🔧 Construyendo interfaz...")
    dashboard = TradingDashboard(df_results, df_trades)
    
    # Mostrar UI
    ui = dashboard.construir_ui()
    display(ui)
    
    # Inicializar visualizaciones
    print("✓ Dashboard listo")
    print("-" * 60)
    print("💡 Tip: Usa los sliders para filtrar estrategias y haz clic en los gráficos para ver la curva de capital\n")
    dashboard._on_filter_change(None)

# Ejecutar
if __name__ == "__main__":
    main()

🚀 Iniciando Dashboard de Análisis de Estrategias v2.0...
------------------------------------------------------------
✓ Datos cargados: 37666 estrategias, 2713 operaciones

🔧 Construyendo interfaz...


✓ Dashboard listo
------------------------------------------------------------
💡 Tip: Usa los sliders para filtrar estrategias y haz clic en los gráficos para ver la curva de capital

